```{python}
from scipy.integrate import quad
import mpmath as mp
import numpy as np
import matplotlib.pyplot as plt
```

# Example 3.6 ergodic doubling map
```{python}
def dbl(x0, n, prec):
    mp.mp.prec = prec

    if isinstance(x0, str):
        x = mp.mpf(x0)
    else:
        x = x0

    one = mp.mpf(1)
    two = mp.mpf(2)

    orbit = [mp.mpf(0)] * n
    orbit[0] = x

    for i in range(1, n):
        x = x * two
        if x >= one:
            x -= one
        orbit[i] = x

    return orbit
```

```{python}
M = 10
n    = 5000
prec = 500000

mp.mp.prec = prec

mean_arr = np.zeros((n,M))
orb = [0] * n
for i in range(M):
    orb = np.array([float(xx) for xx in dbl(mp.rand(), n, prec=prec)])
    mean_arr[:,i] = np.cumsum(orb) / np.arange(1,n+1)

norm_arr = np.zeros(n)
for i in range(n):
    norm_arr[i] = np.sqrt( np.mean((mean_arr[i,:] - 0.5)**2) )
```

## Plot of convergence in $\mathcal{L}_2$
```{python}
plt.figure(figsize=(8,5))
plt.scatter(np.arange(100, n), norm_arr[100:], 
            s=0.5, c='darkred', alpha=0.5)
plt.xlabel('$n$')
plt.ylabel('Distance in $\\mathcal{L}_{2}$')
plt.savefig('/home/lorenz/latex/projects/seminar/figures/dbl_neumann.pdf')
plt.show()
```

## Plot of convergence at different starting values
```{python}
mean_arr.reshape(n*M)
plt.figure(figsize=(8,4))
plt.scatter(np.array([np.arange(n)] * M).reshape(n*M),
            mean_arr.reshape(n*M), c='darkred', alpha=0.8, s = 0.5)
plt.hlines(0.5, 0, n, colors='blue', linestyles='dashed',
           label=r'$E(X)$')
plt.xlabel('n')
plt.ylabel('$X_{n}$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/dbl_birkhoff.pdf')
plt.show()
```

# ergodic Rotation

```{python}
M = 10
intg = np.mean(np.exp(np.arange(0,1, 0.1)))
```

## starting at a single value
```{python}
n = 10000
x0 = 9/10

orb = (x0 + np.arange(n)/M) % 1
running_mean = np.cumsum(np.exp( orb )) / np.arange(1, n+1)
```

Orbit:
```{python}
plt.figure(figsize = (8,4))
plt.scatter(np.arange(20), orb[:20], 
            s=2, c='blue')
plt.xlabel('n')
plt.ylabel('$T^{n}(\omega)$')
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rote_orbit.pdf')
plt.show()
```

Convergence to the expected value:
```{python}
plt.figure(figsize = (8,4))
plt.scatter(np.arange(1, 101), running_mean[:100], 
            s=1, c='darkred', alpha=1)
plt.hlines(intg, 0, 100, colors='blue', linestyles='dashed',
           label=r'$E(X)$', alpha = 0.5)
plt.xlabel('n')
plt.ylabel('$X_{n}(\omega)$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rote_birkhoff.pdf')
plt.show()
```

## starting at 10 different values

```{python}
n = 100000

mean_arr = np.zeros((n,10))
orb = [0] * n
for i in range(10):
    x0 = i / M
    orb = (x0 + np.arange(n)/M) % 1
    mean_arr[:,i] = np.cumsum(np.exp(orb)) / np.arange(1,n+1)

norm_arr = np.zeros(n)
for i in range(n):
    norm_arr[i] = np.sqrt( np.mean((mean_arr[i,:] - intg)**2) )
```

Convergence in $\mathcal{L}_2$:
```{python}
plt.figure(figsize = (8,4))
plt.scatter(np.arange(1,101), norm_arr[:100], 
            s=1, c='darkred', alpha=0.5)
plt.xlabel('n')
plt.ylabel('Distance in $\mathcal{L}_{2}$')
plt.tight_layout()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rote_neumann.pdf')
plt.show()
```

# discrete non-ergodic rotation

```{python}
def rot_d(x0,n):
    orbit = [0] *n
    orbit[0] = x0
    x = x0
    for i in range(1,n):
        x = ( x + 2 ) % 4
        orbit[i] = x
    return orbit
n = 100
orba = np.array(rot_d(0, n))
orbb = np.array(rot_d(1,n))
orbc = np.array(rot_d(2, n))
orbd = np.array(rot_d(3, n))

meana = np.cumsum(orba) / np.arange(1,n+1)
meanb = np.cumsum(orbb) / np.arange(1,n+1)
meanc = np.cumsum(orbc) / np.arange(1,n+1)
meand = np.cumsum(orbd) / np.arange(1,n+1)

norm = (np.abs(meana - 1) + np.abs(meanb - 2) + np.abs(meanc - 1) + np.abs(meand - 2))/4
float(norm[n-2])
```

Orbit:
```{python}
plt.figure(figsize = (8,4))
plt.scatter(np.arange(1, 11), orba[:10], 
            s=50, c='darkred', marker = '^', label = '$\\omega = 0$')
plt.plot(np.arange(1, 11), orba[:10], 
            c='darkred', marker = '^')
plt.scatter(np.arange(1, 11), orbb[:10], 
            s=50, c='forestgreen', label = '$\\omega = 1$')
plt.plot(np.arange(1, 11), orbb[:10], 
            c='forestgreen')
plt.xlabel('$n$')
plt.ylabel('$T^{n}(\\omega)$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rotn_orbit.pdf')
plt.show()
```

Convergence to the expected value:
```{python}
plt.figure(figsize = (8,4))
plt.hlines((1,2), 1, 51, colors='blue', linestyles='dashed',
           label=r'$E(X)$', alpha = 0.3)
plt.scatter(np.arange(1, 51), meana[:50], 
            s=5, c='darkred', alpha=1)
plt.scatter(np.arange(1, 51), meanc[:50], 
            s=5, c='darkred', alpha=1)
plt.scatter(np.arange(1, 51), meanb[:50], 
            s=5, c='green', alpha=1)
plt.scatter(np.arange(1, 51), meand[:50], 
            s=5, c='green', alpha=1)
plt.xlabel('n')
plt.ylabel('$X_{n}$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rotn_birkhoff.pdf')
plt.show()
```

Convergence in $\mathcal{L}_1$:
```{python}
plt.figure(figsize = (8,4))
plt.scatter(np.arange(1, n+1), norm, 
            s=5, c='darkred')
plt.xlabel('n')
plt.ylabel('Distance in \mathcal{L}_{1}$')
plt.savefig('/home/lorenz/latex/projects/seminar/figures/rotn_neumann.pdf')
plt.show()
```

# Two examples of non-measure-preserving transformations: 
First, we consider $T(x) = \frac{x}{2}$.
Using this transformation we do not get convergence:

```{python}
N   = 200
x0  = 0.7

ns = np.arange(N)
xs = x0 * (0.5 ** ns)

time_avg = np.cumsum(xs) / (ns + 1)

plt.figure(figsize=(6,4))
plt.scatter(ns, time_avg, s = 0.5)
plt.hlines(0.5, 0, N-1, colors='red', linestyles='dashed', alpha=0.5,
           label='$\mathbb{E}(X)$')
plt.xlabel('$n$')
plt.ylabel('$X_{n}(\omega)$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/xdiv_birkhoff.png')
plt.show()
```

Same for $T(x) = 1 - \frac{x}{2}$:

```{python}
def l(x0,n):
    orbit = [0] * n
    orbit[0] = x0
    x = x0
    for i in range(1,n):
        x = 1 - x/2
        orbit[i] = x
    return orbit
n = 50
orb = np.array(l(0.0000234, n))
mean = np.cumsum(orb) / np.arange(1,n+1)
```

```{python}
plt.figure(figsize=(8, 4))
plt.scatter(np.arange(1,n+1), mean, s=2)
plt.hlines(0.5, 1, n, colors='red', linestyles='dashed', label='$\mathbb{E}(X)$', alpha=0.5)
plt.xlabel('$n$')
plt.ylabel('$X_{n}(\omega)$')
plt.legend()
plt.savefig('/home/lorenz/latex/projects/seminar/figures/x2_birkhoff.pdf')
plt.show()
```

# Visualizing the application of Birkhoff's ergodic theorem to the Metropolis algorithm 

This function takes in a starting value, a probability matrix, the distribution 
we are interested in, and the length of the sequence to be generated and
returns the corresponding sequence of the Markov chain generated by the Metropolis
algorithm.

```{python}
def metropolis(start,n, q,pi):
    if start not in range(1, q.shape[0]+1): 
        raise Exception('Startwert ist nicht in state space.')
    n_states = q.shape[0]
    r = np.zeros((n_states,n_states+1))
    r[:,1:r.shape[1]] = np.cumsum(q,axis=1)
    u = np.random.uniform(0,1,(2,n))

    seq = np.zeros(n+1, dtype=int)
    seq[0] = start

    for i in range(n):
        state = seq[i] -1
        for j in range(n_states):
            if u[0,i] >= r[state, j] and u[0,i] < r[state,j+1]:
                if u[1,i] < (pi[j]*q[j,state])/(pi[state]*q[state,j]):
                    seq[i+1] = j+1
                else: seq[i+1] = seq[i]
                break
    return seq

def f(x):
    return np.exp(-x**2)
```

We want to sample from the distribution $N e^{-x^2}$ normalized to integrate to $1$
on the unit interval $[0,1]$.

```{python}
m = 101
states = np.linspace(-1, 1, m)
p = np.random.dirichlet(np.ones(m), (m,))
pi = f(states)
```

We always need to pick a starting value between 1 and m!!!

```{python}
n = 100000
s = metropolis(23,n,p,pi)
s_normed = (s - np.arange(1,m+1).mean())/((m-1)/2)
norm_fac = quad(f,-1,1)[0]
```

```{python}
burnin = 500
seq = s_normed[burnin:]
plt.hist(seq, bins=101, density=True)
plt.plot(states, pi/norm_fac, '-', linewidth=4)
plt.savefig('/home/lorenz/latex/projects/seminar/figures/markov.pdf')
plt.show()
```